# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

A content team managing a few thousand live pages cannot manually re-check every page each month. **Can search-visibility and on-page signals rank pages so an editor reviews the ones most likely to be declining first, before the traffic loss compounds?**

- **Unit of analysis:** one content page (`content_id`).
- **Decision supported:** which pages enter this week's editorial review queue — not whether to publish, unpublish, or auto-rewrite a page.
- **Cost of getting the order wrong:** a missed decliner keeps losing impressions silently until someone happens to notice it; a false alarm costs an editor a few minutes checking a page that turns out to be fine. Because the second mistake is cheap and the first is a slow leak, the queue below is designed to rank generously rather than filter aggressively.

In [1]:
print("Decision this notebook supports: rank candidate pages so an editor opens the")
print("highest-priority ones first each week — a ranked review queue, not an auto-publish system.")
print()
print("Unit of analysis : one content page (content_id)")
print("Output           : rank + suggested_action + reason codes")


Decision this notebook supports: rank candidate pages so an editor opens the
highest-priority ones first each week — a ranked review queue, not an auto-publish system.

Unit of analysis : one content page (content_id)
Output           : rank + suggested_action + reason codes


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Anonymized slice of the FlyRank ML-internship warehouse: **30,000 content rows across 32 pseudonymous clients**, with 90-day trailing search performance (impressions, clicks, sessions, average position, CTR, scroll rate) plus on-page signals (word count, content age, days since last update) and 30-day trend windows.

**Excluded on purpose:** titles, URLs, domains, client names, keywords, raw exports — nothing here can identify a client or page.

**Label — `is_declining_label`:** 1 when `trend_direction == "down"` (last-30-day impressions more than 20% below the prior 30 days), else 0.

In [2]:
import pandas as pd

frame = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
print(f"Rows: {len(frame):,}")
print(f"Clients: {frame['client_id'].nunique()}")
print(f"Declining rate (base rate): {frame['is_declining_label'].mean():.1%}")
frame[["client_id", "content_type", "impressions_90d", "avg_position", "is_declining_label"]].head(3)


Rows: 30,000
Clients: 32
Declining rate (base rate): 54.2%


,client_id,content_type,impressions_90d,avg_position,is_declining_label
0,client_f369cb89fc,keyword article,3803,10.6,1
1,client_4e07408562,keyword article,15320,20.3,1
2,client_7f2253d7e2,keyword article,12581,36.5,1


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Baseline:** a transparent, hand-written rule (`scripts/02_baseline_score.py`) that flags pages on readable conditions — stale-but-visible, declining-with-demand, low CTR at high visibility, and similar. Any editor can check it by hand; every model below has to beat it on the *same* split and metric.

**Models compared:** logistic regression, a decision tree, and a random forest (scikit-learn), all trained on the same ~30-feature matrix (numeric traffic/engagement signals + one-hot categorical fields). `trend_direction` and `trend_pct` — the columns that define the label — are excluded from the feature set at the pipeline level, since including either would let the model read the answer off the input.

**Split — client holdout, not random rows:** pages from the same client share structure (template, cadence, niche). A random row-level split can let a model see one page from a client in training and a different page from the *same* client at test time, which leaks client identity into the score. Every result below uses a **client-holdout split**: whole clients are held out of training and scored only at test time.

**Leakage audit:** `impressions_90d` correlates **0.98** with the sum of the label's two 30-day windows, and that 60-day sum covers **56.2%** of the 90-day total on average — confirming the label windows had to stay out of the feature set.

In [3]:
import sys; sys.path.insert(0, "../../scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

print("trend_direction in model features:", "trend_direction" in MODEL_CATEGORICAL_FEATURES)
print("trend_pct in model features:      ", "trend_pct" in MODEL_NUMERIC_FEATURES)

overlap = frame[["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]].copy()
overlap["last60_sum"] = overlap["impressions_last_30d"] + overlap["impressions_prev_30d"]
corr = overlap["impressions_90d"].corr(overlap["last60_sum"])
coverage = (overlap["last60_sum"] / overlap["impressions_90d"].replace(0, float("nan"))).mean()
print(f"\nCorrelation (90d vs sum of label windows): {corr:.2f}")
print(f"Coverage (60d window / 90d total):          {coverage:.1%}")


trend_direction in model features: False
trend_pct in model features:       False

Correlation (90d vs sum of label windows): 0.98
Coverage (60d window / 90d total):          56.2%


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| Model | ROC AUC | Avg. precision | Precision@50 | Recall | F1 |
|---|---|---|---|---|---|
| Baseline rule | 0.627 | 0.468 | 0.240 | — | — |
| Logistic regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| Decision tree | 0.742 | 0.575 | 0.660 | 0.716 | 0.634 |
| **Random forest** | **0.750** | **0.618** | **0.740** | 0.744 | 0.640 |

**Why the split matters more than the model:** the identical random forest scores **0.90 precision@50** under a naive random row split — but 31 of 32 clients leak between train and test under that split. The honest, client-holdout number is **0.74** — still a 3.1x lift over the 24% baseline, and above the 54.2% base rate, but nowhere near 0.90.

*Careful language:* observed — on a client-holdout test set, the model's top-50 ranked pages contained a declining page 74% of the time. This is decision-support for review prioritization, not a causal or guaranteed prediction for any single page, and it says nothing about Google's ranking algorithm.

In [4]:
import json

results = json.load(open("../../outputs/model_results.json"))
for name, m in results.get("models", {}).items():
    print(f"{name:20s}  ROC AUC={m.get('roc_auc', 0):.3f}  P@50={m.get('precision_at_50', 0):.3f}")
print("\nBest model    :", results.get("best_model", {}).get("name"))
print("Split strategy:", results.get("split_strategy"))


decision_tree         ROC AUC=0.742  P@50=0.660
logistic_regression   ROC AUC=0.700  P@50=0.400
random_forest         ROC AUC=0.750  P@50=0.740

Best model    : random_forest
Split strategy: client_holdout


## 5. Limitations

*What this work cannot claim.*

- **Observational, not causal.** The model finds pages that *look like* past decliners — it does not explain *why*, and refreshing a flagged page is not guaranteed to reverse the trend.
- **32 clients is a small holdout universe.** Client-level validation is the honest choice, but a handful of unusual clients can move the headline number more than in a larger sample.
- **A large share of the queue is low-confidence** — those rows are a hint to look, not an instruction to act.
- **One action bucket (`expand_and_refresh`) is too small to generalize from** — treat it as a lead worth a manual look, not a validated pattern.
- **Anonymized starter slice, not the full warehouse** — numbers here will shift on the full internship dataset.
- No client or page identifiers appear anywhere in this notebook or the deployed paper.

In [5]:
queue = pd.read_csv("../../outputs/refresh_queue.csv")
print("Confidence tiers:")
print(queue["confidence"].value_counts())
print()
print("Smallest action bucket:")
print(queue["suggested_action"].value_counts().tail(1))


Confidence tiers:
confidence
low       15000
medium    11398
high       3602
Name: count, dtype: int64

Smallest action bucket:
suggested_action
expand_and_refresh    82
Name: count, dtype: int64


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Each of the 30,000 pages gets a rank, a confidence tier, a `suggested_action`, and plain-language `reason_codes` (e.g. `declining_with_demand`, `low_ctr_visible_page`) so an editor can check the reasoning before acting — not just trust a number. Intended use: an editor opens high-confidence rows first, checks the page against its reason codes, then decides whether to refresh, expand, or leave it. Low-confidence rows are a hint to look, not an instruction to act.

In [6]:
action_mix = queue["suggested_action"].value_counts()
print("Action mix across all 30,000 pages:")
print(action_mix)
print()
print("Top of the queue:")
cols = [c for c in ["final_rank", "final_refresh_score", "suggested_action", "confidence", "final_reason_codes"] if c in queue.columns]
queue.sort_values("final_rank").head(5)[cols]


Action mix across all 30,000 pages:
suggested_action
monitor                          13083
refresh                           8188
refresh_and_review_ctr            6654
refresh_and_review_engagement     1993
expand_and_refresh                  82
Name: count, dtype: int64

Top of the queue:


,final_rank,final_refresh_score,suggested_action,confidence,final_reason_codes
0,1,81.734212,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|low...
1,2,81.603243,refresh_and_review_ctr,medium,declining_with_demand|low_ctr_visible_page|mod...
2,3,81.544618,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
3,4,81.169731,refresh_and_review_ctr,high,declining_with_demand|low_ctr_visible_page|mod...
4,5,80.957565,refresh_and_review_ctr,medium,declining_with_demand|low_ctr_visible_page|mod...


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Everything the deployed research paper cites is generated by this repo's pipeline (`scripts/run_all.py`) and lives under `outputs/` — regenerating it from a clean clone reproduces every number in this notebook and in the paper.

In [7]:
import os

print("outputs/model_results.json:", os.path.exists("../../outputs/model_results.json"))
print("outputs/refresh_queue.csv :", os.path.exists("../../outputs/refresh_queue.csv"))
print("outputs/model_report.md  :", os.path.exists("../../outputs/model_report.md"))
print("\nCharts:")
for f in sorted(os.listdir("../../outputs/charts")):
    print(" -", f)


outputs/model_results.json: True
outputs/refresh_queue.csv : True
outputs/model_report.md  : True

Charts:
 - action_mix.svg
 - confidence_mix.svg
 - top_feature_importance.svg
 - top_reason_codes.svg
 - trend_distribution.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 8. Demo outline — 5-minute walkthrough (Week-8 showcase)

*Question -> Method -> One chart -> One honest result -> One recommendation. Timed for 5 minutes.*

**1. Question (45 sec)**
A content team with a few thousand live pages can't manually re-check each one every month.
Which pages should an editor open first? This capstone builds a ranked review queue for that
decision, not an auto-publish/auto-fix system.

**2. Method (60 sec)**
30,000 anonymized FlyRank content records across 32 clients. Compared a transparent rule-based
baseline against logistic regression, a decision tree, and a random forest. Validated on a
**client-holdout split** — no client's pages appear in both train and test — because that is the
split that matches how this would actually be used (on clients the model hasn't seen yet).

**3. One chart (60 sec)**
Show `outputs/charts/top_feature_importance.svg` (or the client-holdout vs naive-split precision
comparison in the paper's Results section). One message: the model beats the baseline, but the
*size* of that win depends entirely on which split you trust.

**4. One honest result (60 sec)**
Random forest: **Precision@50 = 0.74** on the client-holdout split — a genuine 3.1x lift over the
24% baseline rule, and above the 54% base rate. The same model reads as **0.90 precision@50**
under a naive random split that let 31 of 32 clients leak between train and test. That gap
(0.90 vs 0.74) is the paper's central finding, not a footnote.

**5. One recommendation (45 sec)**
Ship the client-holdout number as the honest one, and always report which split a claim came
from — the split, not the model, was the bigger lever on the headline metric here.

---

## 9. Two shareable cuts

*Repurposing the same work for two audiences — Week 7's session covered how to frame each.*

**Short social post (methodology-focused):**

> Trained 3 models to rank which content pages need review first — but the real finding wasn't
> the model. It was the validation split. A naive random split said 90% precision@50. A
> client-holdout split (the honest one, since a real system meets new clients) said 74%. Same
> model, same data, 16-point gap — just from letting the split leak. Full writeup + repo linked.

**Employer-facing 3-sentence summary:**

> I built a content-review-priority model on 30,000 anonymized FlyRank records across 32 clients,
> comparing a rule-based baseline against logistic regression, a decision tree, and a random
> forest. Using a client-holdout validation split, the random forest reached 0.74 precision@50 —
> a 3.1x lift over the baseline — versus a misleadingly high 0.90 under a naive split with client
> leakage. The project's core contribution is that validation-design finding, packaged as a
> ranked, reason-coded review queue with monitoring triggers for when it goes stale.
